In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2013.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2012.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2014.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2009.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2007.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2015.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2006.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2010.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2008.nc
/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday/era5_daily_2006_2015/era5_daily_2011.nc
/kaggle/in

In [3]:
# ── Cell 1 : imports + load data ─────────────────────────────────────────────
import numpy as np
import pandas as pd
import xarray as xr
from scipy import ndimage
from scipy import stats
import subprocess
import glob, os, warnings
warnings.filterwarnings("ignore")

# install mann-kendall
subprocess.run(['pip', 'install', 'pymannkendall', '--quiet'], check=True)
import pymannkendall as mk

# install regionmask
subprocess.run(['pip', 'install', 'regionmask', '--quiet'], check=True)
import regionmask

print("imports done ✓")

# ── load ERA5 data ──
DATA_DIR = '/kaggle/input/datasets/divyanshyecho/era5-0-25-daily-mmday'
nc_files  = sorted(glob.glob(f'{DATA_DIR}/**/*.nc', recursive=True))
print(f"found {len(nc_files)} files")

ds = xr.open_mfdataset(nc_files, combine='by_coords',
                        engine='h5netcdf', chunks=None)

data_precip = ds['tp'].values.astype('float32')
lat         = ds['latitude'].values.astype('float32')
lon         = ds['longitude'].values.astype('float32')
time        = ds['time'].values
times_pd    = pd.to_datetime(time)

print(f"shape : {data_precip.shape}")
print(f"time  : {str(times_pd[0])[:10]} → {str(times_pd[-1])[:10]}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.3/72.3 kB 979.7 kB/s eta 0:00:00
imports done ✓
found 46 files
shape : (5612, 141, 161)
time  : 1979-06-01 → 2024-09-30


In [4]:
# ── Cell 2 : India mask + threshold ──────────────────────────────────────────
RAINY_DAY_MIN = 1.0
PERCENTILE    = 99.0
FLOOR_MM      = 50.0

# India mask
countries  = regionmask.defined_regions.natural_earth_v5_0_0.countries_110
india_raw  = countries.mask(lon, lat)
india_mask = (india_raw.values == 98)
print(f"India cells : {india_mask.sum()} / {141*161}")

# mask — India cells + rainy days only
data_masked = np.where(
    (data_precip > RAINY_DAY_MIN) & india_mask[np.newaxis, :, :],
    data_precip, np.nan
).astype('float32')

# threshold
jjas_mask  = (times_pd.month >= 6) & (times_pd.month <= 9)
jjas_data  = data_masked[jjas_mask, :, :]
per_grid_99 = np.nanpercentile(jjas_data, PERCENTILE, axis=0).astype('float32')
per_grid_threshold = np.maximum(per_grid_99, FLOOR_MM).astype('float32')
per_grid_threshold[~india_mask] = 999.0

# exceedance mask
data_precip_safe = np.where(np.isnan(data_precip), 0.0, data_precip).astype('float32')
threshold_3d     = np.broadcast_to(
    per_grid_threshold[np.newaxis, :, :], data_precip.shape
).copy().astype('float32')
c = (data_precip_safe > threshold_3d)

print(f"exceedance mask shape   : {c.shape}")
print(f"overall exceedance rate : {c.mean()*100:.3f}%")
print("threshold done ✓")

India cells : 4452 / 22701
exceedance mask shape   : (5612, 141, 161)
overall exceedance rate : 0.114%
threshold done ✓


In [5]:
from scipy import ndimage

struct8 = ndimage.generate_binary_structure(2, 2)   # 8-connectivity everywhere

n_days = data_precip.shape[0]   # 5612
n_lat  = len(lat)               # 141
n_lon  = len(lon)               # 161

Label8  = np.zeros((n_days, n_lat, n_lon), dtype='int32')
NE8_raw = np.zeros(n_days, dtype='int32')
NT8_raw = np.zeros(n_days, dtype='int32')

for k in range(n_days):
    img        = c[k].astype('int8')
    NT8_raw[k] = int(img.sum())
    lab, nr    = ndimage.label(img, structure=struct8)
    Label8[k]  = lab
    NE8_raw[k] = nr
    if k % 500 == 0:
        print(f"  labeled day {k:4d}/{n_days}")

print(f"\nmean NE = {NE8_raw.mean():.3f} objects/day  (expected ~2.434)")
print(f"mean NT = {NT8_raw.mean():.3f} cells/day    (expected ~25.93)")
print("8-connectivity labeling done ✓")

  labeled day    0/5612
  labeled day  500/5612
  labeled day 1000/5612
  labeled day 1500/5612
  labeled day 2000/5612
  labeled day 2500/5612
  labeled day 3000/5612
  labeled day 3500/5612
  labeled day 4000/5612
  labeled day 4500/5612
  labeled day 5000/5612
  labeled day 5500/5612

mean NE = 2.434 objects/day  (expected ~2.434)
mean NT = 25.929 cells/day    (expected ~25.93)
8-connectivity labeling done ✓


In [6]:
import os

# ── hyperparameters ──────────────────────────────────────────────────────────
MIN_TRACK_CELLS   = 5
MIN_TREND_CELLS   = 16
MUTUAL_STRUCTURAL = 0.10
MUTUAL_CONT       = 0.05
SEARCH_DEG        = 2.5
SIZE_RATIO_LO     = 0.4
SIZE_RATIO_HI     = 3.0
GAP_TOLERANCE     = 1
DOMAIN_EXIT_FRAC  = 0.50

OUT = '/kaggle/working/'
os.makedirs(OUT, exist_ok=True)

# ── output containers ────────────────────────────────────────────────────────
track_stats_rows = []
merge_event_rows = []
split_event_rows = []

# ── global counters ──────────────────────────────────────────────────────────
global_track_num        = 0
global_merge_id         = 0
global_split_id         = 0
dormancy_events_counter = 0

# ── active track registry ────────────────────────────────────────────────────
active_tracks = {}

# ── birth-order counter per date (for reliable track_id NNN suffix) ──────────
births_per_date = {}

# ── set of June-1 date strings for season_start detection (Bug 5 fix) ────────
first_jjas_days = {
    pd.Timestamp(f'{yr}-06-01').strftime('%Y%m%d')
    for yr in range(1979, 2025)
}

print("hyperparameters and global variables initialised ✓")
print(f"first_jjas_days sample : {sorted(first_jjas_days)[:3]}  …")

hyperparameters and global variables initialised ✓
first_jjas_days sample : ['19790601', '19800601', '19810601']  …


In [7]:
def jjas_day_num(ts):
    """Jun1=1, Jun30=30, Jul1=31, Sep30=122."""
    return int((ts - pd.Timestamp(f'{ts.year}-06-01')).days + 1)


def get_objects(day_idx, min_cells=MIN_TRACK_CELLS):
    """
    Return {lbl: {size, centroid:(lat,lon), outside}} for all objects
    with size >= min_cells on day_idx.
    """
    labeled = Label8[day_idx]
    n_obj   = NE8_raw[day_idx]
    objs    = {}
    for lbl in range(1, n_obj + 1):
        mask = (labeled == lbl)
        sz   = int(mask.sum())
        if sz < min_cells:
            continue
        rows_, cols_ = np.where(mask)
        clat    = float(lat[rows_].mean())
        clon    = float(lon[cols_].mean())
        outside = int((mask & ~india_mask).sum())
        objs[lbl] = {'size': sz, 'centroid': (clat, clon), 'outside': outside}
    return objs


def compute_overlaps(day_t, day_t1, objs_t, objs_t1):
    """
    Compute all overlapping (i,j) pairs between day_t and day_t1.
    Called ONCE per day-pair — no threshold applied here.
    Returns {(lbl_i, lbl_j): (fwd, bwd, shared, mutual)}.
    mutual = min(fwd, bwd) — the core CP5 symmetric measure.
    """
    lab_t  = Label8[day_t]
    lab_t1 = Label8[day_t1]
    pairs  = {}
    for lbl_i, obj_i in objs_t.items():
        mask_i         = (lab_t == lbl_i)
        overlap_labels = np.unique(lab_t1[mask_i])
        overlap_labels = overlap_labels[overlap_labels > 0]
        for lbl_j in overlap_labels:
            if lbl_j not in objs_t1:
                continue
            shared = int((mask_i & (lab_t1 == lbl_j)).sum())
            if shared == 0:
                continue
            fwd    = shared / obj_i['size']
            bwd    = shared / objs_t1[lbl_j]['size']
            mutual = min(fwd, bwd)
            pairs[(lbl_i, lbl_j)] = (fwd, bwd, shared, mutual)
    return pairs


print("helper functions defined ✓")

helper functions defined ✓


In [8]:
def new_track(day_idx, lbl, obj, start_type, split_from=-1):
    """Create a new track and return its integer track_num."""
    global global_track_num
    global_track_num += 1
    tnum = global_track_num

    ts       = times_pd[day_idx]
    date_str = ts.strftime('%Y%m%d')

    births_per_date[date_str] = births_per_date.get(date_str, 0) + 1
    born_today = births_per_date[date_str]
    tid = f'{date_str}_{born_today:03d}'

    clat, clon = obj['centroid']

    row = {
        'track_num'        : tnum,
        'track_id'         : tid,
        'date'             : date_str,
        'year'             : ts.year,
        'jjas_day'         : jjas_day_num(ts),
        'day_of_track'     : 0,
        'size'             : obj['size'],
        'centroid_lat'     : round(clat, 3),
        'centroid_lon'     : round(clon, 3),
        'dormant'          : False,
        'merge_event'      : False,
        'split_event'      : False,
        'start_type'       : start_type,
        'end_type'         : None,
        'duration'         : None,
        'split_from_track' : split_from,
        'merge_into_track' : -1,
        'day_lbl'          : lbl,
    }

    active_tracks[tnum] = {
        'track_id'       : tid,
        'birth_date'     : date_str,
        'last_day'       : day_idx,
        'last_lbl'       : lbl,
        'last_centroid'  : obj['centroid'],
        'last_size'      : obj['size'],
        'dormant_days'   : 0,
        'start_type'     : start_type,
        'duration'       : 1,
        'split_from'     : split_from,
        'n_merge_events' : 0,
        'n_split_events' : 0,
        'daily_rows'     : [row],
    }
    return tnum


def append_track_day(tnum, day_idx, lbl, obj,
                     dormant=False, merge_event=False, split_event=False):
    """
    Append one daily row. Caller must update last_day/last_lbl/
    last_centroid/last_size after calling this for non-dormant rows.
    """
    tr = active_tracks[tnum]
    tr['duration'] += 1

    ts         = times_pd[day_idx]
    sz         = obj['size'] if not dormant else 0
    clat, clon = obj['centroid'] if not dormant else (None, None)

    if not dormant:
        tr['dormant_days'] = 0
        if merge_event: tr['n_merge_events'] += 1
        if split_event: tr['n_split_events'] += 1

    row = {
        'track_num'        : tnum,
        'track_id'         : tr['track_id'],
        'date'             : ts.strftime('%Y%m%d'),
        'year'             : ts.year,
        'jjas_day'         : jjas_day_num(ts),
        'day_of_track'     : tr['duration'] - 1,
        'size'             : sz,
        'centroid_lat'     : round(clat, 3) if clat is not None else None,
        'centroid_lon'     : round(clon, 3) if clon is not None else None,
        'dormant'          : dormant,
        'merge_event'      : merge_event,
        'split_event'      : split_event,
        'start_type'       : tr['start_type'],
        'end_type'         : None,
        'duration'         : None,
        'split_from_track' : tr['split_from'],
        'merge_into_track' : -1,
        'day_lbl'          : lbl if not dormant else -1,
    }
    tr['daily_rows'].append(row)


def terminate_track(tnum, end_type, merge_into=-1):
    """
    Finalise track. Strips trailing dormant rows on natural_death.
    If no rows remain after stripping, track is silently discarded.
    Fills duration on all rows, end_type on last row only.
    """
    if tnum not in active_tracks:
        return
    tr   = active_tracks[tnum]
    rows = tr['daily_rows']
    dur  = tr['duration']

    if end_type == 'natural_death':
        while rows and rows[-1]['dormant']:
            rows.pop()
            dur -= 1
        if not rows:
            del active_tracks[tnum]
            return

    for row in rows:
        row['duration'] = dur

    rows[-1]['end_type']         = end_type
    rows[-1]['merge_into_track'] = merge_into

    track_stats_rows.extend(rows)
    del active_tracks[tnum]


print("track management functions defined ✓")

track management functions defined ✓


In [9]:
# ── Cell 8 : main tracking loop (corrected — Jun 1 fix applied) ──────────────

import time as _time

print("starting main tracking loop …")
_t0 = _time.time()

# ── PRE-LOOP: birth Jun 1 1979 objects ───────────────────────────────────────
# Jun 1 1979 is day_idx=0. It is always objs_today, never objs_next.
# Step 7 only births objs_next, so Jun 1 1979 would otherwise be silently missed.
objs_day0 = get_objects(0)
for lbl_j, obj_j in objs_day0.items():
    tnum_new = new_track(0, lbl_j, obj_j, 'season_start')
    if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
        terminate_track(tnum_new, 'domain_exit')
print(f"pre-loop: Jun 1 1979 — {len(objs_day0)} season_start objects born")

for day_idx in range(n_days - 1):

    ts_today = times_pd[day_idx]
    ts_next  = times_pd[day_idx + 1]

    # ── September 30: hard season termination ────────────────────────────────
    # After terminating, birth Jun 1 of the next season before continuing.
    # Reason: the pair (Sep 30, Jun 1) is skipped by continue, so Jun 1 objects
    # in objs_next would otherwise never reach Step 7.
    if ts_today.month == 9 and ts_today.day == 30:
        for tnum in list(active_tracks.keys()):
            terminate_track(tnum, 'season_end')
        # birth Jun 1 of next season as season_start
        if day_idx + 1 < n_days:
            objs_jun1 = get_objects(day_idx + 1)
            for lbl_j, obj_j in objs_jun1.items():
                tnum_new = new_track(day_idx + 1, lbl_j, obj_j, 'season_start')
                if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
                    terminate_track(tnum_new, 'domain_exit')
        continue

    objs_today = get_objects(day_idx)
    objs_next  = get_objects(day_idx + 1)

    # ── Bug 2 fix: two separate early-exit cases ──────────────────────────────

    # Case A: nothing on T+1 → all active tracks go dormant; no births
    if len(objs_next) == 0:
        for tnum in list(active_tracks.keys()):
            tr = active_tracks.get(tnum)
            if tr is None: continue
            if tr['dormant_days'] < GAP_TOLERANCE:
                tr['dormant_days'] += 1
                dormancy_events_counter += 1
                dummy = {'size': 0, 'centroid': tr['last_centroid']}
                append_track_day(tnum, day_idx + 1, -1, dummy, dormant=True)
            else:
                terminate_track(tnum, 'natural_death')
        continue

    # Case B: nothing on T → all active tracks go dormant; birth T+1 objects
    if len(objs_today) == 0:
        for tnum in list(active_tracks.keys()):
            tr = active_tracks.get(tnum)
            if tr is None: continue
            if tr['dormant_days'] < GAP_TOLERANCE:   # Bug 7 fix
                tr['dormant_days'] += 1
                dormancy_events_counter += 1
                dummy = {'size': 0, 'centroid': tr['last_centroid']}
                append_track_day(tnum, day_idx + 1, -1, dummy, dormant=True)
            else:
                terminate_track(tnum, 'natural_death')
        for lbl_j, obj_j in objs_next.items():
            stype    = ('season_start' if ts_next.strftime('%Y%m%d') in first_jjas_days
                        else 'natural_birth')
            tnum_new = new_track(day_idx + 1, lbl_j, obj_j, stype)
            if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
                terminate_track(tnum_new, 'domain_exit')
        continue

    # ── STEP 1: compute all overlaps once, no threshold ───────────────────────
    pairs = compute_overlaps(day_idx, day_idx + 1, objs_today, objs_next)

    fwd_links = {}
    bwd_links = {}
    for (lbl_i, lbl_j), (fwd, bwd, shared, mutual) in pairs.items():
        fwd_links.setdefault(lbl_i, []).append((lbl_j, fwd, bwd, shared, mutual))
        bwd_links.setdefault(lbl_j, []).append((lbl_i, fwd, bwd, shared, mutual))

    # Bug 1 fix: no dormant_days==0 filter
    lbl_to_track = {
        tr['last_lbl']: tnum
        for tnum, tr in active_tracks.items()
        if tr['last_day'] == day_idx
    }

    handled_today = set()
    handled_next  = set()

    # ── STEP 2: MERGE detection ───────────────────────────────────────────────
    merge_candidates = []
    for lbl_j in objs_next:
        parents_raw = bwd_links.get(lbl_j, [])
        genuine = [
            (lbl_i, fwd, bwd, shared, mutual)
            for (lbl_i, fwd, bwd, shared, mutual) in parents_raw
            if mutual >= MUTUAL_STRUCTURAL and lbl_i in lbl_to_track
        ]
        if len(genuine) >= 2:
            genuine.sort(key=lambda x: x[4], reverse=True)
            genuine  = genuine[:2]
            combined = genuine[0][4] + genuine[1][4]
            merge_candidates.append((combined, lbl_j, genuine))

    merge_candidates.sort(key=lambda x: x[0], reverse=True)

    for combined, lbl_j, genuine in merge_candidates:
        if lbl_j in handled_next: continue
        if any(g[0] in handled_today for g in genuine): continue
        genuine_valid = [g for g in genuine if lbl_to_track.get(g[0]) in active_tracks]
        if len(genuine_valid) < 2: continue
        genuine = genuine_valid[:2]

        genuine.sort(key=lambda x: objs_today[x[0]]['size'], reverse=True)
        lbl_dom, fwd_dom, bwd_dom, sh_dom, mut_dom = genuine[0]
        lbl_abs, fwd_abs, bwd_abs, sh_abs, mut_abs = genuine[1]
        tnum_dom = lbl_to_track[lbl_dom]
        tnum_abs = lbl_to_track[lbl_abs]
        obj_j    = objs_next[lbl_j]
        dom_size = objs_today[lbl_dom]['size']
        abs_size = objs_today[lbl_abs]['size']
        clat_j, clon_j = obj_j['centroid']

        global_merge_id += 1
        merge_event_rows.append({
            'merge_id'             : global_merge_id,
            'date'                 : ts_next.strftime('%Y%m%d'),
            'year'                 : ts_next.year,
            'jjas_day'             : jjas_day_num(ts_next),
            'dominant_track'       : tnum_dom,
            'absorbed_tracks'      : str(tnum_abs),
            'n_absorbed'           : 1,
            'dominant_size_before' : dom_size,
            'absorbed_size_before' : abs_size,
            'merged_size_after'    : obj_j['size'],
            'mutual_dominant'      : round(mut_dom, 4),
            'mutual_absorbed'      : round(mut_abs, 4),
            'centroid_lat'         : round(clat_j, 3),
            'centroid_lon'         : round(clon_j, 3),
            'inside_CI'            : (15 <= clat_j <= 25) and (75 <= clon_j <= 85),
            'merge_type'           : 'merge' if abs_size >= 5 else 'absorption',
        })

        append_track_day(tnum_dom, day_idx + 1, lbl_j, obj_j, merge_event=True)
        active_tracks[tnum_dom]['last_day']      = day_idx + 1
        active_tracks[tnum_dom]['last_lbl']      = lbl_j
        active_tracks[tnum_dom]['last_centroid'] = obj_j['centroid']
        active_tracks[tnum_dom]['last_size']     = obj_j['size']

        terminate_track(tnum_abs, 'merge_death', merge_into=tnum_dom)
        handled_today.add(lbl_dom)
        handled_today.add(lbl_abs)
        handled_next.add(lbl_j)

    # ── STEP 3: SPLIT detection ───────────────────────────────────────────────
    for lbl_i, obj_i in objs_today.items():
        if lbl_i in handled_today: continue
        if lbl_i not in lbl_to_track: continue
        tnum_par = lbl_to_track[lbl_i]
        if tnum_par not in active_tracks: continue

        children_raw = fwd_links.get(lbl_i, [])
        genuine = [
            (lbl_j, fwd, bwd, shared, mutual)
            for (lbl_j, fwd, bwd, shared, mutual) in children_raw
            if mutual >= MUTUAL_STRUCTURAL and lbl_j not in handled_next
        ]
        if len(genuine) < 2: continue

        genuine.sort(key=lambda x: x[4], reverse=True)
        genuine = genuine[:2]
        genuine.sort(key=lambda x: objs_next[x[0]]['size'], reverse=True)

        lbl_dom_d, fwd_dom_d, bwd_dom_d, sh_dom_d, mut_dom_d = genuine[0]
        lbl_chd_d, fwd_chd_d, bwd_chd_d, sh_chd_d, mut_chd_d = genuine[1]
        obj_dom = objs_next[lbl_dom_d]
        obj_chd = objs_next[lbl_chd_d]

        sum_fwd       = fwd_dom_d + fwd_chd_d
        lost_fraction = round(max(0.0, 1.0 - sum_fwd), 4)
        size_asymm    = round(obj_dom['size'] / (obj_dom['size'] + obj_chd['size']), 4)
        clat_i, clon_i = obj_i['centroid']

        append_track_day(tnum_par, day_idx + 1, lbl_dom_d, obj_dom, split_event=True)
        active_tracks[tnum_par]['last_day']      = day_idx + 1
        active_tracks[tnum_par]['last_lbl']      = lbl_dom_d
        active_tracks[tnum_par]['last_centroid'] = obj_dom['centroid']
        active_tracks[tnum_par]['last_size']     = obj_dom['size']
        # Bug A fix: no explicit n_split_events increment — append_track_day did it

        tnum_chd = new_track(day_idx + 1, lbl_chd_d, obj_chd,
                             'split_birth', split_from=tnum_par)
        if obj_chd['outside'] / obj_chd['size'] > DOMAIN_EXIT_FRAC:
            terminate_track(tnum_chd, 'domain_exit')

        global_split_id += 1
        split_event_rows.append({
            'split_id'          : global_split_id,
            'date'              : ts_next.strftime('%Y%m%d'),
            'year'              : ts_next.year,
            'jjas_day'          : jjas_day_num(ts_next),
            'parent_track'      : tnum_par,
            'dominant_daughter' : tnum_par,
            'child_track_1'     : tnum_chd,
            'child_track_2'     : -1,
            'n_children'        : 1,
            'parent_size'       : obj_i['size'],
            'dominant_size'     : obj_dom['size'],
            'child_size_1'      : obj_chd['size'],
            'child_size_2'      : -1,
            'mutual_dominant'   : round(mut_dom_d, 4),
            'mutual_child_1'    : round(mut_chd_d, 4),
            'mutual_child_2'    : -1,
            'size_asymmetry'    : size_asymm,
            'lost_fraction'     : lost_fraction,
            'centroid_lat'      : round(clat_i, 3),
            'centroid_lon'      : round(clon_i, 3),
            'inside_CI'         : (15 <= clat_i <= 25) and (75 <= clon_i <= 85),
        })

        handled_today.add(lbl_i)
        handled_next.add(lbl_dom_d)
        handled_next.add(lbl_chd_d)

    # ── STEP 4: CONTINUATION (greedy, mutual >= MUTUAL_CONT) ─────────────────
    cont_candidates = []
    for (lbl_i, lbl_j), (fwd, bwd, shared, mutual) in pairs.items():
        if lbl_i in handled_today or lbl_j in handled_next: continue
        if lbl_i not in lbl_to_track: continue
        if mutual < MUTUAL_CONT: continue
        tnum = lbl_to_track[lbl_i]
        if tnum not in active_tracks: continue
        cont_candidates.append((mutual, lbl_i, lbl_j))

    cont_candidates.sort(key=lambda x: x[0], reverse=True)

    for mutual, lbl_i, lbl_j in cont_candidates:
        if lbl_i in handled_today or lbl_j in handled_next: continue
        tnum = lbl_to_track.get(lbl_i)
        if tnum is None or tnum not in active_tracks: continue
        obj_j = objs_next[lbl_j]

        if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
            terminate_track(tnum, 'domain_exit')
            handled_today.add(lbl_i)
            handled_next.add(lbl_j)
            continue

        append_track_day(tnum, day_idx + 1, lbl_j, obj_j)
        active_tracks[tnum]['last_day']      = day_idx + 1
        active_tracks[tnum]['last_lbl']      = lbl_j
        active_tracks[tnum]['last_centroid'] = obj_j['centroid']
        active_tracks[tnum]['last_size']     = obj_j['size']
        handled_today.add(lbl_i)
        handled_next.add(lbl_j)

    # ── STEPS 5+6: FALLBACK + DORMANCY ───────────────────────────────────────
    for tnum in list(active_tracks.keys()):
        tr = active_tracks.get(tnum)
        if tr is None: continue

        active_today_unlinked = (tr['last_day'] == day_idx and
                                 tr['last_lbl'] not in handled_today)
        currently_dormant     = (tr['dormant_days'] > 0 and
                                 tr['last_day'] < day_idx)

        if not active_today_unlinked and not currently_dormant:
            continue

        clat_i, clon_i = tr['last_centroid']
        size_i         = tr['last_size']
        best_lbl_j     = None
        best_dist      = float('inf')

        for lbl_j, obj_j in objs_next.items():
            if lbl_j in handled_next: continue
            clat_j, clon_j = obj_j['centroid']
            dist = ((clat_j - clat_i)**2 + (clon_j - clon_i)**2) ** 0.5
            if dist > SEARCH_DEG: continue
            if size_i <= 0: continue
            ratio = obj_j['size'] / size_i
            if not (SIZE_RATIO_LO <= ratio <= SIZE_RATIO_HI): continue
            if dist < best_dist:
                best_dist  = dist
                best_lbl_j = lbl_j

        if best_lbl_j is not None:
            obj_j = objs_next[best_lbl_j]
            if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
                terminate_track(tnum, 'domain_exit')
                handled_next.add(best_lbl_j)
            else:
                append_track_day(tnum, day_idx + 1, best_lbl_j, obj_j)
                active_tracks[tnum]['last_day']      = day_idx + 1
                active_tracks[tnum]['last_lbl']      = best_lbl_j
                active_tracks[tnum]['last_centroid'] = obj_j['centroid']
                active_tracks[tnum]['last_size']     = obj_j['size']
                handled_next.add(best_lbl_j)
        else:
            tr = active_tracks.get(tnum)
            if tr is None: continue
            if tr['dormant_days'] < GAP_TOLERANCE:
                tr['dormant_days'] += 1
                dormancy_events_counter += 1
                dummy = {'size': 0, 'centroid': tr['last_centroid']}
                append_track_day(tnum, day_idx + 1, -1, dummy, dormant=True)
            else:
                terminate_track(tnum, 'natural_death')

    # ── STEP 7: BIRTHS ────────────────────────────────────────────────────────
    for lbl_j, obj_j in objs_next.items():
        if lbl_j in handled_next: continue
        # Note: ts_next is never Jun 1 in Step 7 (Jun 1 is handled in pre-loop
        # and Sep 30 block above). This step handles Jun 2 onwards.
        stype    = ('season_start' if ts_next.strftime('%Y%m%d') in first_jjas_days
                    else 'natural_birth')
        tnum_new = new_track(day_idx + 1, lbl_j, obj_j, stype)
        if obj_j['outside'] / obj_j['size'] > DOMAIN_EXIT_FRAC:
            terminate_track(tnum_new, 'domain_exit')

    if day_idx % 500 == 0:
        elapsed = _time.time() - _t0
        print(f"  day {day_idx:4d}/{n_days-1}  active={len(active_tracks):4d}  "
              f"rows={len(track_stats_rows):6d}  elapsed={elapsed:.1f}s")

# post-loop: catches Sep 30 2024 tracks
for tnum in list(active_tracks.keys()):
    terminate_track(tnum, 'season_end')

elapsed = _time.time() - _t0
print(f"\ntracking loop complete — {elapsed:.1f}s")
print(f"total track_stats_rows  : {len(track_stats_rows):,}")
print(f"total merge_event_rows  : {len(merge_event_rows):,}")
print(f"total split_event_rows  : {len(split_event_rows):,}")
print(f"dormancy events         : {dormancy_events_counter:,}")

starting main tracking loop …
pre-loop: Jun 1 1979 — 0 season_start objects born
  day 1500/5611  active=   3  rows=  1682  elapsed=0.6s
  day 2000/5611  active=   3  rows=  2312  elapsed=0.8s
  day 2500/5611  active=   4  rows=  2999  elapsed=1.0s
  day 4000/5611  active=   3  rows=  4767  elapsed=1.7s
  day 4500/5611  active=   1  rows=  5358  elapsed=1.9s
  day 5000/5611  active=   2  rows=  5924  elapsed=2.1s

tracking loop complete — 2.4s
total track_stats_rows  : 6,767
total merge_event_rows  : 23
total split_event_rows  : 16
dormancy events         : 4,297


In [10]:
# ── Cell 9 : build DataFrames ─────────────────────────────────────────────────

df_stats = pd.DataFrame(track_stats_rows)
df_merge = pd.DataFrame(merge_event_rows) if merge_event_rows else pd.DataFrame()
df_split = pd.DataFrame(split_event_rows) if split_event_rows else pd.DataFrame()

# ── track summary — one row per track ────────────────────────────────────────
summary_rows = []
for tnum, grp in df_stats.groupby('track_num'):
    active_rows = grp[~grp['dormant']]
    first_row   = grp.iloc[0]
    last_row    = grp.iloc[-1]
    summary_rows.append({
        'track_num'        : tnum,
        'track_id'         : first_row['track_id'],
        'birth_date'       : first_row['date'],
        'death_date'       : last_row['date'],
        'duration'         : (int(first_row['duration'])
                              if pd.notna(first_row['duration'])
                              else len(grp)),
        'peak_size'        : int(active_rows['size'].max()) if len(active_rows) > 0 else 0,
        'mean_size'        : (round(float(active_rows['size'].mean()), 2)
                              if len(active_rows) > 0 else 0.0),
        'birth_lat'        : first_row['centroid_lat'],
        'birth_lon'        : first_row['centroid_lon'],
        'start_type'       : first_row['start_type'],
        'end_type'         : last_row['end_type'],
        'split_from_track' : first_row['split_from_track'],
        'merge_into_track' : last_row['merge_into_track'],
        'n_merge_events'   : int(grp['merge_event'].sum()),
        'n_split_events'   : int(grp['split_event'].sum()),
    })

df_summary = pd.DataFrame(summary_rows)

# ── track id lookup ───────────────────────────────────────────────────────────
df_lookup = (df_stats[['track_num', 'track_id']]
             .drop_duplicates()
             .reset_index(drop=True))
df_lookup['birth_date'] = df_lookup['track_id'].str[:8]

print("dataframes built ✓")
print(f"df_stats   : {df_stats.shape}")
print(f"df_summary : {df_summary.shape}")
print(f"df_merge   : {df_merge.shape}")
print(f"df_split   : {df_split.shape}")

dataframes built ✓
df_stats   : (6767, 18)
df_summary : (4162, 15)
df_merge   : (23, 16)
df_split   : (16, 21)


In [11]:
# ── Cell 10 : build daily label array ────────────────────────────────────────

date_to_idx = {times_pd[k].strftime('%Y%m%d'): k for k in range(n_days)}

print("building TrackLabel array …")
TrackLabel = np.zeros((n_days, n_lat, n_lon), dtype='int32')

for _, row in df_stats.iterrows():
    if row['dormant'] or row['size'] == 0:
        continue
    dlbl = row['day_lbl']
    if dlbl <= 0:
        continue
    tnum    = int(row['track_num'])
    day_idx = date_to_idx.get(row['date'], -1)
    if day_idx < 0:
        continue
    TrackLabel[day_idx][Label8[day_idx] == dlbl] = tnum

nonzero = int((TrackLabel > 0).sum())
print(f"TrackLabel built — shape {TrackLabel.shape}")
print(f"non-zero cells : {nonzero:,}   (expected ~130,604)")

building TrackLabel array …
TrackLabel built — shape (5612, 141, 161)
non-zero cells : 130,929   (expected ~130,604)


In [12]:
# ── Cell 11 : save all outputs ────────────────────────────────────────────────

df_stats.to_csv(f'{OUT}track_statistics_long.csv', index=False)
df_summary.to_csv(f'{OUT}track_summary.csv', index=False)
df_lookup.to_csv(f'{OUT}track_id_lookup.csv', index=False)

if len(df_merge) > 0:
    df_merge.to_csv(f'{OUT}merge_events.csv', index=False)
if len(df_split) > 0:
    df_split.to_csv(f'{OUT}split_events.csv', index=False)

np.save(f'{OUT}TrackLabel_1979_2024.npy', TrackLabel)

print("all outputs saved ✓")
for fname in ['track_statistics_long.csv', 'track_summary.csv',
              'merge_events.csv', 'split_events.csv',
              'track_id_lookup.csv', 'TrackLabel_1979_2024.npy']:
    path = f'{OUT}{fname}'
    if os.path.exists(path):
        mb = os.path.getsize(path) / 1e6
        print(f"  {fname:<38} {mb:7.1f} MB")

all outputs saved ✓
  track_statistics_long.csv                  0.7 MB
  track_summary.csv                          0.4 MB
  merge_events.csv                           0.0 MB
  split_events.csv                           0.0 MB
  track_id_lookup.csv                        0.1 MB
  TrackLabel_1979_2024.npy                 509.6 MB


In [13]:
# ── Cell 12 : summary statistics ─────────────────────────────────────────────

print('=' * 58)
print('CP5 TRACKING SUMMARY')
print('=' * 58)
print(f"total tracks            : {df_summary.shape[0]:,}")
print(f"total merge events      : {len(df_merge):,}")
print(f"total split events      : {len(df_split):,}")
print(f"total stat rows         : {df_stats.shape[0]:,}")
print(f"dormancy events         : {dormancy_events_counter:,}")

print("\nTrack start types:")
print(df_summary['start_type'].value_counts().to_string())

print("\nTrack end types:")
print(df_summary['end_type'].value_counts().to_string())

print("\nDuration by peak size class:")
for label, lo, hi in [('Sub-ref  (5-15) ', 5,  15),
                       ('Medium  (16-90) ', 16, 90),
                       ('Large   (>=91)  ', 91, 9999)]:
    grp = df_summary[df_summary['peak_size'].between(lo, hi)]
    if len(grp) == 0:
        continue
    print(f"  {label} — n={len(grp):,}")
    print(f"    mean duration : {grp['duration'].mean():.2f} days")
    print(f"    median        : {grp['duration'].median():.1f} days")
    print(f"    % single-day  : {(grp['duration']==1).mean()*100:.1f}%")

years = np.arange(1979, 2025)
if len(df_merge) > 0:
    yr_merge = df_merge.groupby('year').size().reindex(years, fill_value=0)
    print(f"\nMerge : mean={yr_merge.mean():.2f}/yr  "
          f"std={yr_merge.std():.2f}  "
          f"range {yr_merge.min()}-{yr_merge.max()}")
if len(df_split) > 0:
    yr_split = df_split.groupby('year').size().reindex(years, fill_value=0)
    print(f"Split : mean={yr_split.mean():.2f}/yr  "
          f"std={yr_split.std():.2f}  "
          f"range {yr_split.min()}-{yr_split.max()}")

ml = df_summary[df_summary['peak_size'] >= 16]
print(f"\nMedium+Large tracks ({len(ml):,}):")
print(f"  with >=1 merge event  : {(ml['n_merge_events']>0).sum()}")
print(f"  with >=1 split event  : {(ml['n_split_events']>0).sum()}")
print(f"  born as split_birth   : {(ml['start_type']=='split_birth').sum()}")
print(f"  died as merge_death   : {(ml['end_type']=='merge_death').sum()}")

CP5 TRACKING SUMMARY
total tracks            : 4,162
total merge events      : 23
total split events      : 16
total stat rows         : 6,767
dormancy events         : 4,297

Track start types:
start_type
natural_birth    4128
season_start       18
split_birth        16

Track end types:
end_type
natural_death    4109
season_end         30
merge_death        23

Duration by peak size class:
  Sub-ref  (5-15)  — n=2,497
    mean duration : 1.31 days
    median        : 1.0 days
    % single-day  : 77.7%
  Medium  (16-90)  — n=1,560
    mean duration : 2.04 days
    median        : 2.0 days
    % single-day  : 39.4%
  Large   (>=91)   — n=105
    mean duration : 3.10 days
    median        : 3.0 days
    % single-day  : 14.3%

Merge : mean=0.50/yr  std=0.69  range 0-3
Split : mean=0.35/yr  std=0.60  range 0-2

Medium+Large tracks (1,665):
  with >=1 merge event  : 22
  with >=1 split event  : 16
  born as split_birth   : 5
  died as merge_death   : 11


In [14]:
# ── Cell 15 : mutual score distribution diagnostic ────────────────────────────
# Reproduces the Section 8 diagnostic. Run after tracking to compare
# pre-tracking raw candidates vs what was actually captured.

print("=== MUTUAL SCORE DISTRIBUTION DIAGNOSTIC ===\n")

all_mutual_scores = []
merge_candidates_by_threshold = {t: 0 for t in [0.05,0.08,0.10,0.12,0.15,0.18,0.20,0.25,0.30]}

sample_days = range(0, n_days - 1, 5)   # every 5th day for speed

for day_idx in sample_days:
    ts = times_pd[day_idx]
    if ts.month == 9 and ts.day == 30:
        continue
    objs_t  = get_objects(day_idx)
    objs_t1 = get_objects(day_idx + 1)
    if not objs_t or not objs_t1:
        continue
    pairs_raw = compute_overlaps(day_idx, day_idx + 1, objs_t, objs_t1)

    bwd_raw = {}
    for (lbl_i, lbl_j), (fwd, bwd, shared, mutual) in pairs_raw.items():
        bwd_raw.setdefault(lbl_j, []).append(mutual)
        all_mutual_scores.append(mutual)

    for lbl_j, mutuals in bwd_raw.items():
        if len(mutuals) >= 2:
            sorted_m = sorted(mutuals, reverse=True)
            second_m = sorted_m[1]
            for thresh in merge_candidates_by_threshold:
                if second_m >= thresh:
                    merge_candidates_by_threshold[thresh] += 1

all_m = np.array(all_mutual_scores)
print(f"Sampled {len(sample_days)} days ({len(all_m):,} overlap pairs)\n")
print("Overall mutual score distribution:")
for p in [25, 50, 75, 90, 95, 99]:
    print(f"  {p}th pct : {np.percentile(all_m, p):.4f}")
print(f"  mean     : {all_m.mean():.4f}")
print(f"  % >= MUTUAL_CONT ({MUTUAL_CONT})       : {(all_m >= MUTUAL_CONT).mean()*100:.1f}%")
print(f"  % >= MUTUAL_STRUCTURAL ({MUTUAL_STRUCTURAL}) : {(all_m >= MUTUAL_STRUCTURAL).mean()*100:.1f}%")

scale = n_days / (len(sample_days) * 5)   # rough scale-up
print(f"\nMerge candidates (2nd parent >= threshold) — scaled to full dataset:")
print(f"{'Threshold':>12}  {'Sampled':>10}  {'~Full Dataset':>15}  {'~per yr':>10}")
for thresh, cnt in merge_candidates_by_threshold.items():
    full = cnt * scale
    print(f"  >= {thresh:.2f}    {cnt:>10}  {full:>15.1f}  {full/46:>10.1f}")

=== MUTUAL SCORE DISTRIBUTION DIAGNOSTIC ===

Sampled 1123 days (373 overlap pairs)

Overall mutual score distribution:
  25th pct : 0.1111
  50th pct : 0.1944
  75th pct : 0.3333
  90th pct : 0.5000
  95th pct : 0.5900
  99th pct : 0.7015
  mean     : 0.2406
  % >= MUTUAL_CONT (0.05)       : 90.3%
  % >= MUTUAL_STRUCTURAL (0.1) : 76.9%

Merge candidates (2nd parent >= threshold) — scaled to full dataset:
   Threshold     Sampled    ~Full Dataset     ~per yr
  >= 0.05            10             10.0         0.2
  >= 0.08             8              8.0         0.2
  >= 0.10             5              5.0         0.1
  >= 0.12             3              3.0         0.1
  >= 0.15             1              1.0         0.0
  >= 0.18             0              0.0         0.0
  >= 0.20             0              0.0         0.0
  >= 0.25             0              0.0         0.0
  >= 0.30             0              0.0         0.0


In [15]:
# ── Cell 16 : step-level link attribution ────────────────────────────────────
# Reconstructs what fraction of continuations came from each step.
# Uses df_stats: a track row that is NOT dormant, NOT merge_event, NOT split_event
# and has day_of_track > 0 is a continuation. We can't distinguish Step 4 vs 5
# from df_stats alone, but we can measure the dormancy revival rate (Step 6 outcome).

print("=== STEP-LEVEL LINK ATTRIBUTION ===\n")

total_rows      = len(df_stats)
dormant_rows    = df_stats['dormant'].sum()
merge_rows      = df_stats['merge_event'].sum()
split_rows      = df_stats['split_event'].sum()
birth_rows      = (df_stats['day_of_track'] == 0).sum()
continuation    = total_rows - dormant_rows - birth_rows

print(f"Total rows          : {total_rows:,}")
print(f"  Birth rows        : {birth_rows:,}  ({birth_rows/total_rows*100:.1f}%)")
print(f"  Dormant rows      : {dormant_rows:,}  ({dormant_rows/total_rows*100:.1f}%)")
print(f"  Continuation rows : {continuation:,}  ({continuation/total_rows*100:.1f}%)")
print(f"    of which merge  : {merge_rows:,}  ({merge_rows/continuation*100:.1f}% of cont)")
print(f"    of which split  : {split_rows:,}  ({split_rows/continuation*100:.1f}% of cont)")

# Dormancy revival rate
n_dormancy_revived = dormancy_events_counter - df_summary['duration'].sub(
    df_stats.groupby('track_num').size()
).clip(lower=0).sum()  # approximation

print(f"\nDormancy events total : {dormancy_events_counter:,}")

# Track efficiency: what % of track-days are 'useful' (active, not dormant)
active_track_days = df_stats[~df_stats['dormant']]
print(f"\nTrack utilisation (non-dormant days / total days): "
      f"{len(active_track_days)/total_rows*100:.1f}%")

# Average gap between dormant and revival per track
tracks_with_dormancy = df_stats.groupby('track_num')['dormant'].any()
n_tracks_ever_dormant = tracks_with_dormancy.sum()
print(f"Tracks that were ever dormant : {n_tracks_ever_dormant} "
      f"({n_tracks_ever_dormant/len(df_summary)*100:.1f}% of all tracks)")

=== STEP-LEVEL LINK ATTRIBUTION ===

Total rows          : 6,767
  Birth rows        : 4,162  (61.5%)
  Dormant rows      : 188  (2.8%)
  Continuation rows : 2,417  (35.7%)
    of which merge  : 23  (1.0% of cont)
    of which split  : 16  (0.7% of cont)

Dormancy events total : 4,297

Track utilisation (non-dormant days / total days): 97.2%
Tracks that were ever dormant : 179 (4.3% of all tracks)


In [16]:
# ── Cell 17 : merge/split spatiotemporal pattern ─────────────────────────────

print("=== MERGE / SPLIT SPATIOTEMPORAL ANALYSIS ===\n")

if len(df_merge) > 0:
    print("MERGES")
    print(f"  Total       : {len(df_merge)}")
    print(f"  Inside CI   : {df_merge['inside_CI'].sum()}  ({df_merge['inside_CI'].mean()*100:.0f}%)")
    print(f"  Type 'merge': {(df_merge['merge_type']=='merge').sum()}")
    print(f"  Type 'absorb': {(df_merge['merge_type']=='absorption').sum()}")
    print(f"\n  Size stats (cells):")
    print(f"    dominant_before : mean={df_merge['dominant_size_before'].mean():.1f}  "
          f"median={df_merge['dominant_size_before'].median():.0f}")
    print(f"    absorbed_before : mean={df_merge['absorbed_size_before'].mean():.1f}  "
          f"median={df_merge['absorbed_size_before'].median():.0f}")
    print(f"    merged_after    : mean={df_merge['merged_size_after'].mean():.1f}  "
          f"median={df_merge['merged_size_after'].median():.0f}")
    growth = df_merge['merged_size_after'] / (df_merge['dominant_size_before'] + df_merge['absorbed_size_before'])
    print(f"\n  Growth factor (merged / sum_parents): mean={growth.mean():.2f}  "
          f"median={growth.median():.2f}")
    print(f"  Median > 1.0 means sub-threshold intensification is real.")
    print(f"\n  Mutual score — absorbed (2nd parent):")
    print(f"    mean={df_merge['mutual_absorbed'].mean():.4f}  "
          f"median={df_merge['mutual_absorbed'].median():.4f}  "
          f"min={df_merge['mutual_absorbed'].min():.4f}")
    print(f"\n  Monthly distribution:")
    df_merge['month'] = pd.to_datetime(df_merge['date'], format='%Y%m%d').dt.month
    print(df_merge['month'].value_counts().sort_index().rename({6:'Jun',7:'Jul',8:'Aug',9:'Sep'}).to_string())
    print(f"\n  Yearly: mean={len(df_merge)/46:.2f}/yr")
else:
    print("No merge events — check MUTUAL_STRUCTURAL threshold.")

if len(df_split) > 0:
    print("\nSPLITS")
    print(f"  Total       : {len(df_split)}")
    print(f"  Inside CI   : {df_split['inside_CI'].sum()}  ({df_split['inside_CI'].mean()*100:.0f}%)")
    print(f"  Mean lost_fraction : {df_split['lost_fraction'].mean():.3f}")
    print(f"  Mean size_asymmetry: {df_split['size_asymmetry'].mean():.3f}  "
          f"(1.0=all dominant, 0.5=equal)")
    print(f"\n  Yearly: mean={len(df_split)/46:.2f}/yr")
else:
    print("No split events — check MUTUAL_STRUCTURAL threshold.")

=== MERGE / SPLIT SPATIOTEMPORAL ANALYSIS ===

MERGES
  Total       : 23
  Inside CI   : 6  (26%)
  Type 'merge': 23
  Type 'absorb': 0

  Size stats (cells):
    dominant_before : mean=46.1  median=30
    absorbed_before : mean=18.9  median=11
    merged_after    : mean=75.3  median=48

  Growth factor (merged / sum_parents): mean=1.46  median=1.16
  Median > 1.0 means sub-threshold intensification is real.

  Mutual score — absorbed (2nd parent):
    mean=0.1589  median=0.1556  min=0.1000

  Monthly distribution:
month
Jun    7
Jul    7
Aug    6
Sep    3

  Yearly: mean=0.50/yr

SPLITS
  Total       : 16
  Inside CI   : 4  (25%)
  Mean lost_fraction : 0.599
  Mean size_asymmetry: 0.658  (1.0=all dominant, 0.5=equal)

  Yearly: mean=0.35/yr


In [17]:
# ── Cell 20 : JJAS intra-season profile ──────────────────────────────────────
# Shows how ERE frequency, NE, and NT vary across the JJAS season.
# Useful for checking the Jun 1 artefact is truly gone.

print("=== JJAS INTRA-SEASON DAY PROFILE ===\n")

jjas_day_NE  = np.zeros(122)
jjas_day_NT  = np.zeros(122)
jjas_day_cnt = np.zeros(122, dtype=int)

for k in range(n_days):
    ts  = times_pd[k]
    jd  = int((ts - pd.Timestamp(f'{ts.year}-06-01')).days)  # 0-indexed
    if jd < 0 or jd >= 122: continue

    # Only >=MIN_TREND_CELLS objects
    labeled = Label8[k]
    n_obj   = NE8_raw[k]
    ne, nt  = 0, 0
    for lbl in range(1, n_obj + 1):
        sz = int((labeled == lbl).sum())
        if sz >= MIN_TREND_CELLS:
            ne += 1
            nt += sz
    jjas_day_NE[jd]  += ne
    jjas_day_NT[jd]  += nt
    jjas_day_cnt[jd] += 1

mean_NE_by_day = np.where(jjas_day_cnt > 0, jjas_day_NE / jjas_day_cnt, 0)
mean_NT_by_day = np.where(jjas_day_cnt > 0, jjas_day_NT / jjas_day_cnt, 0)

print("Day  Date   mean_NE  mean_NT")
months = {0:'Jun', 30:'Jul', 61:'Aug', 92:'Sep'}
for jd in range(122):
    base = pd.Timestamp('2000-06-01') + pd.Timedelta(days=jd)
    label = f"{base.strftime('%b%d')}"
    if jd in [0, 1, 29, 30, 60, 61, 91, 92, 121]:
        print(f"  {jd+1:3d}  {label}  NE={mean_NE_by_day[jd]:.3f}  NT={mean_NT_by_day[jd]:.3f}"
              f"  {'<-- Jun 1 check' if jd == 0 else ''}")

# Check Jun 1 specifically — should not be anomalously high or low
jun1_ne = mean_NE_by_day[0]
jun2_ne = mean_NE_by_day[1]
ratio = jun1_ne / jun2_ne if jun2_ne > 0 else 0
print(f"\nJun 1 / Jun 2 NE ratio: {ratio:.3f}  "
      f"{'✓ OK (close to 1)' if 0.5 <= ratio <= 2.0 else '⚠ POSSIBLE ARTEFACT'}")

=== JJAS INTRA-SEASON DAY PROFILE ===

Day  Date   mean_NE  mean_NT
    1  Jun01  NE=0.109  NT=4.457  <-- Jun 1 check
    2  Jun02  NE=0.217  NT=4.587  
   30  Jun30  NE=0.609  NT=23.478  
   31  Jul01  NE=0.478  NT=20.543  
   61  Jul31  NE=0.543  NT=16.022  
   62  Aug01  NE=0.717  NT=25.457  
   92  Aug31  NE=0.370  NT=13.435  
   93  Sep01  NE=0.348  NT=14.609  
  122  Sep30  NE=0.087  NT=6.587  

Jun 1 / Jun 2 NE ratio: 0.500  ✓ OK (close to 1)


In [18]:
# Merge threshold diagnostic — paste as new cell after Cell 4

import numpy as np
import pandas as pd

all_top2_mutual = []
all_top2_fwd    = []
all_top2_bwd    = []
all_growth      = []

for day_idx in range(0, n_days - 1):
    ts = times_pd[day_idx]
    if ts.month == 9 and ts.day == 30:
        continue
    if NE8_raw[day_idx] == 0 or NE8_raw[day_idx + 1] == 0:
        continue

    lab_t  = Label8[day_idx]
    lab_t1 = Label8[day_idx + 1]

    # build size dicts
    objs_t  = {l: int((lab_t  == l).sum()) for l in range(1, NE8_raw[day_idx]   + 1) if int((lab_t  == l).sum()) >= 5}
    objs_t1 = {l: int((lab_t1 == l).sum()) for l in range(1, NE8_raw[day_idx+1] + 1) if int((lab_t1 == l).sum()) >= 5}
    if not objs_t or not objs_t1:
        continue

    # overlaps: bwd_scores[lbl_j] = list of (mutual, fwd, bwd, size_i)
    bwd_scores = {}
    for lbl_i, sz_i in objs_t.items():
        mask_i = (lab_t == lbl_i)
        for lbl_j in np.unique(lab_t1[mask_i]):
            if lbl_j == 0 or lbl_j not in objs_t1:
                continue
            shared = int((mask_i & (lab_t1 == lbl_j)).sum())
            if shared == 0:
                continue
            fwd = shared / sz_i
            bwd = shared / objs_t1[lbl_j]
            bwd_scores.setdefault(lbl_j, []).append((min(fwd,bwd), fwd, bwd, sz_i))

    for lbl_j, parents in bwd_scores.items():
        if len(parents) < 2:
            continue
        parents.sort(reverse=True)
        p1_sz = parents[0][3]
        p2_sz = parents[1][3]
        all_top2_mutual.append(parents[1][0])
        all_top2_fwd.append(parents[1][1])
        all_top2_bwd.append(parents[1][2])
        all_growth.append(objs_t1[lbl_j] / (p1_sz + p2_sz))

top2 = np.array(all_top2_mutual)
top2f = np.array(all_top2_fwd)
top2b = np.array(all_top2_bwd)
gf    = np.array(all_growth)
N     = len(top2)

print(f"Total candidates (2+ parents, no threshold): {N}  (~{N/46:.1f}/yr)\n")

print("── 2nd-parent mutual score percentiles ──")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p}th pct : {np.percentile(top2, p):.4f}")
print(f"  mean    : {top2.mean():.4f}")

print("\n── Threshold sweep (AND = mutual >= t) ──")
print(f"  {'thresh':>8}  {'count':>8}  {'%total':>8}  {'/yr':>8}")
for t in [0.03, 0.05, 0.07, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]:
    n = (top2 >= t).sum()
    mark = " ◄ current" if t == 0.10 else ""
    print(f"  {t:>8.2f}  {n:>8}  {n/N*100:>7.1f}%  {n/46:>8.1f}{mark}")

print("\n── OR logic: max(fwd,bwd) >= t ──")
or_score = np.maximum(top2f, top2b)
print(f"  {'thresh':>8}  {'AND':>8}  {'OR':>8}  {'extra':>8}")
for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    n_and = (top2 >= t).sum()
    n_or  = (or_score >= t).sum()
    print(f"  {t:>8.2f}  {n_and:>8}  {n_or:>8}  +{n_or-n_and:<7}")

print("\n── Bottleneck: what limits top2 mutual ──")
print(f"  BWD limits (bwd < fwd): {(top2b < top2f).sum()} ({(top2b < top2f).sum()/N*100:.1f}%)")
print(f"  FWD limits (fwd <= bwd): {(top2f <= top2b).sum()} ({(top2f <= top2b).sum()/N*100:.1f}%)")

print("\n── Growth factor: merged_size / (p1+p2) ──")
print(f"  mean={gf.mean():.2f}  median={np.median(gf):.2f}  >1.0: {(gf>1.0).sum()/N*100:.0f}%  >1.5: {(gf>1.5).sum()/N*100:.0f}%  >2.0: {(gf>2.0).sum()/N*100:.0f}%")

print(f"\n── To match Nikumbh ~1.4/yr ──")
target = 1.4 * 46
for t in np.arange(0.01, 0.51, 0.01):
    if (top2 >= t).sum() <= target:
        print(f"  AND threshold needed: ~{t:.2f}  (gives {(top2>=t).sum()/46:.1f}/yr)")
        break
for t in np.arange(0.01, 0.51, 0.01):
    if (or_score >= t).sum() <= target:
        print(f"  OR  threshold needed: ~{t:.2f}  (gives {(or_score>=t).sum()/46:.1f}/yr)")
        break

Total candidates (2+ parents, no threshold): 87  (~1.9/yr)

── 2nd-parent mutual score percentiles ──
  10th pct : 0.0164
  25th pct : 0.0320
  50th pct : 0.0556
  75th pct : 0.1038
  90th pct : 0.1497
  95th pct : 0.1756
  99th pct : 0.2145
  mean    : 0.0729

── Threshold sweep (AND = mutual >= t) ──
    thresh     count    %total       /yr
      0.03        66     75.9%       1.4
      0.05        49     56.3%       1.1
      0.07        36     41.4%       0.8
      0.08        32     36.8%       0.7
      0.10        23     26.4%       0.5 ◄ current
      0.12        17     19.5%       0.4
      0.15         9     10.3%       0.2
      0.20         2      2.3%       0.0
      0.25         1      1.1%       0.0
      0.30         0      0.0%       0.0

── OR logic: max(fwd,bwd) >= t ──
    thresh       AND        OR     extra
      0.05        49        82  +33     
      0.10        23        69  +46     
      0.15         9        64  +55     
      0.20         2        54  +52 

In [19]:
# Split threshold diagnostic — paste as new cell

import numpy as np
import pandas as pd

all_top2_mutual = []
all_top2_fwd    = []
all_top2_bwd    = []
all_growth      = []  # here: parent_size / sum(daughters) — shrinkage factor

for day_idx in range(0, n_days - 1):
    ts = times_pd[day_idx]
    if ts.month == 9 and ts.day == 30:
        continue
    if NE8_raw[day_idx] == 0 or NE8_raw[day_idx + 1] == 0:
        continue

    lab_t  = Label8[day_idx]
    lab_t1 = Label8[day_idx + 1]

    objs_t  = {l: int((lab_t  == l).sum()) for l in range(1, NE8_raw[day_idx]   + 1) if int((lab_t  == l).sum()) >= 5}
    objs_t1 = {l: int((lab_t1 == l).sum()) for l in range(1, NE8_raw[day_idx+1] + 1) if int((lab_t1 == l).sum()) >= 5}
    if not objs_t or not objs_t1:
        continue

    # fwd_scores[lbl_i] = list of (mutual, fwd, bwd, size_j)
    # For splits: one T parent → multiple T+1 daughters
    fwd_scores = {}
    for lbl_i, sz_i in objs_t.items():
        mask_i = (lab_t == lbl_i)
        for lbl_j in np.unique(lab_t1[mask_i]):
            if lbl_j == 0 or lbl_j not in objs_t1:
                continue
            sz_j   = objs_t1[lbl_j]
            shared = int((mask_i & (lab_t1 == lbl_j)).sum())
            if shared == 0:
                continue
            fwd    = shared / sz_i
            bwd    = shared / sz_j
            fwd_scores.setdefault(lbl_i, []).append((min(fwd,bwd), fwd, bwd, sz_j))

    for lbl_i, daughters in fwd_scores.items():
        if len(daughters) < 2:
            continue
        daughters.sort(reverse=True)
        p1_sz = daughters[0][3]
        p2_sz = daughters[1][3]
        all_top2_mutual.append(daughters[1][0])
        all_top2_fwd.append(daughters[1][1])   # shared / size(parent)
        all_top2_bwd.append(daughters[1][2])   # shared / size(daughter)
        # shrinkage: parent_size / sum of daughters (< 1 means parent shrank)
        all_growth.append(objs_t[lbl_i] / (p1_sz + p2_sz))

top2  = np.array(all_top2_mutual)
top2f = np.array(all_top2_fwd)   # fwd = shared / parent  (how much of parent went to this daughter)
top2b = np.array(all_top2_bwd)   # bwd = shared / daughter (how much of daughter came from parent)
gf    = np.array(all_growth)
N     = len(top2)

print(f"Total split candidates (2+ daughters, no threshold): {N}  (~{N/46:.1f}/yr)\n")

print("── 2nd-daughter mutual score percentiles ──")
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p}th pct : {np.percentile(top2, p):.4f}")
print(f"  mean    : {top2.mean():.4f}")

print("\n── Threshold sweep (AND = mutual >= t) ──")
print(f"  {'thresh':>8}  {'count':>8}  {'%total':>8}  {'/yr':>8}")
for t in [0.03, 0.05, 0.07, 0.08, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]:
    n = (top2 >= t).sum()
    mark = " ◄ current" if t == 0.10 else ""
    print(f"  {t:>8.2f}  {n:>8}  {n/N*100:>7.1f}%  {n/46:>8.1f}{mark}")

print("\n── OR logic: max(fwd,bwd) >= t ──")
or_score = np.maximum(top2f, top2b)
print(f"  {'thresh':>8}  {'AND':>8}  {'OR':>8}  {'extra':>8}")
for t in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    n_and = (top2 >= t).sum()
    n_or  = (or_score >= t).sum()
    print(f"  {t:>8.2f}  {n_and:>8}  {n_or:>8}  +{n_or-n_and:<7}")

print("\n── Bottleneck: what limits top2 mutual ──")
# For splits:
# FWD = shared/parent — if low, parent barely contributed to this daughter
# BWD = shared/daughter — if low, daughter grew a lot from other sources
print(f"  FWD limits (fwd < bwd): {(top2f < top2b).sum()} ({(top2f < top2b).sum()/N*100:.1f}%)")
print(f"  BWD limits (bwd < fwd): {(top2b < top2f).sum()} ({(top2b < top2f).sum()/N*100:.1f}%)")
print(f"\n  → FWD = shared/parent_size")
print(f"    If FWD is bottleneck: parent is large, daughter got small fraction of parent")
print(f"  → BWD = shared/daughter_size")
print(f"    If BWD is bottleneck: daughter grew from sub-threshold cells (same as merge problem)")

print("\n── Shrinkage factor: parent_size / sum(daughters) ──")
print(f"  (< 1.0 means daughters together are larger than parent — sub-threshold growth)")
print(f"  (> 1.0 means parent shrank — dissipation during split)")
print(f"  mean={gf.mean():.2f}  median={np.median(gf):.2f}")
print(f"  <0.5: {(gf<0.5).sum()/N*100:.0f}%  "
      f"0.5-1.0: {((gf>=0.5)&(gf<1.0)).sum()/N*100:.0f}%  "
      f"1.0-2.0: {((gf>=1.0)&(gf<2.0)).sum()/N*100:.0f}%  "
      f">2.0: {(gf>=2.0).sum()/N*100:.0f}%")

print("\n── FWD score distribution for 2nd daughter ──")
print(f"  (shared / parent_size — how much of parent went to 2nd daughter)")
for p in [10, 25, 50, 75, 90]:
    print(f"  {p}th pct : {np.percentile(top2f, p):.4f}")
print(f"  mean : {top2f.mean():.4f}")
print(f"  % >= 0.10 : {(top2f >= 0.10).sum()/N*100:.1f}%")
print(f"  % >= 0.15 : {(top2f >= 0.15).sum()/N*100:.1f}%")

print("\n── BWD score distribution for 2nd daughter ──")
print(f"  (shared / daughter_size — how much of daughter came from parent)")
for p in [10, 25, 50, 75, 90]:
    print(f"  {p}th pct : {np.percentile(top2b, p):.4f}")
print(f"  mean : {top2b.mean():.4f}")
print(f"  % >= 0.10 : {(top2b >= 0.10).sum()/N*100:.1f}%")
print(f"  % >= 0.15 : {(top2b >= 0.15).sum()/N*100:.1f}%")

print(f"\n── To match Nikumbh ~1.4/yr ──")
target = 1.4 * 46
for t in np.arange(0.01, 0.51, 0.01):
    if (top2 >= t).sum() <= target:
        print(f"  AND threshold needed: ~{t:.2f}  (gives {(top2>=t).sum()/46:.1f}/yr)")
        break
for t in np.arange(0.01, 0.51, 0.01):
    if (or_score >= t).sum() <= target:
        print(f"  OR  threshold needed: ~{t:.2f}  (gives {(or_score>=t).sum()/46:.1f}/yr)")
        break

Total split candidates (2+ daughters, no threshold): 69  (~1.5/yr)

── 2nd-daughter mutual score percentiles ──
  10th pct : 0.0155
  25th pct : 0.0267
  50th pct : 0.0625
  75th pct : 0.0962
  90th pct : 0.1473
  95th pct : 0.1695
  99th pct : 0.2297
  mean    : 0.0710

── Threshold sweep (AND = mutual >= t) ──
    thresh     count    %total       /yr
      0.03        49     71.0%       1.1
      0.05        41     59.4%       0.9
      0.07        31     44.9%       0.7
      0.08        23     33.3%       0.5
      0.10        17     24.6%       0.4 ◄ current
      0.12        11     15.9%       0.2
      0.15         7     10.1%       0.2
      0.20         2      2.9%       0.0
      0.25         1      1.4%       0.0
      0.30         0      0.0%       0.0

── OR logic: max(fwd,bwd) >= t ──
    thresh       AND        OR     extra
      0.05        41        65  +24     
      0.10        17        54  +37     
      0.15         7        42  +35     
      0.20         2      

In [20]:
# ERE composition analysis — paste as new cell after Cell 3 (labeling done)
# Requires: Label8, NE8_raw, times_pd, n_days, india_mask, lat, lon

import numpy as np
import pandas as pd

MIN_CELLS = 5

# storage
rows = []

for day_idx in range(n_days):
    ts    = times_pd[day_idx]
    lab   = Label8[day_idx]
    n_obj = NE8_raw[day_idx]
    if n_obj == 0:
        continue
    for lbl in range(1, n_obj + 1):
        mask = (lab == lbl)
        sz   = int(mask.sum())
        if sz < MIN_CELLS:
            continue
        rows.append({
            'date'  : ts.strftime('%Y%m%d'),
            'year'  : ts.year,
            'month' : ts.month,
            'size'  : sz,
        })

df = pd.DataFrame(rows)
print(f"Total ERE object-days (>= {MIN_CELLS} cells): {len(df):,}")
print(f"Date range: {df.date.min()} to {df.date.max()}\n")

# ── 1. Overall size distribution ─────────────────────────────────────────────
print("=" * 55)
print("1. OVERALL SIZE DISTRIBUTION (cells per ERE object)")
print("=" * 55)

size_bins   = [5,8,11,16,26,51,91,151,251,501,9999]
size_labels = ['5-7','8-10','11-15','16-25','26-50',
               '51-90','91-150','151-250','251-500','>500']

print(f"\n  {'Size range':>12}  {'Count':>8}  {'%':>7}  {'mean NT contrib':>16}")
for i in range(len(size_bins)-1):
    lo, hi  = size_bins[i], size_bins[i+1]
    mask_sz = (df['size'] >= lo) & (df['size'] < hi)
    cnt     = mask_sz.sum()
    pct     = cnt / len(df) * 100
    mean_sz = df.loc[mask_sz, 'size'].mean() if cnt > 0 else 0
    print(f"  {size_labels[i]:>12}  {cnt:>8,}  {pct:>6.1f}%  {mean_sz:>16.1f}")

print(f"\n  Total objects : {len(df):,}")
print(f"  Total NT cells: {df['size'].sum():,}")
print(f"  Mean size     : {df['size'].mean():.2f} cells")
print(f"  Median size   : {df['size'].median():.1f} cells")
print(f"  Max size      : {df['size'].max()} cells")

for p in [75, 90, 95, 99]:
    print(f"  {p}th pct size  : {np.percentile(df['size'], p):.0f} cells")

# ── 2. Size class summary matching briefing thresholds ───────────────────────
print("\n" + "=" * 55)
print("2. THREE SIZE CLASSES (briefing thresholds)")
print("=" * 55)

classes = [
    ('Sub-ref  (5-15)',   5,  16),
    ('Medium (16-90)',   16,  91),
    ('Large   (>=91)',   91, 9999),
]
print(f"\n  {'Class':>18}  {'Count':>8}  {'%objects':>10}  {'%NT cells':>10}")
total_NT = df['size'].sum()
for label, lo, hi in classes:
    m   = (df['size'] >= lo) & (df['size'] < hi)
    cnt = m.sum()
    nt  = df.loc[m, 'size'].sum()
    print(f"  {label:>18}  {cnt:>8,}  {cnt/len(df)*100:>9.1f}%  {nt/total_NT*100:>9.1f}%")

# ── 3. NE and NT per month ────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("3. NE AND NT PER MONTH (averaged over all years)")
print("=" * 55)

month_names = {6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep'}

# count unique days per month across all years for averaging
for mnum, mname in month_names.items():
    mdf      = df[df['month'] == mnum]
    n_days_m = times_pd[(pd.DatetimeIndex(times_pd).month == mnum)].shape[0]
    NE_total = len(mdf)           # total object-days
    NT_total = mdf['size'].sum()  # total cells
    NE_day   = NE_total / n_days_m
    NT_day   = NT_total / n_days_m
    sbar     = NT_total / NE_total if NE_total > 0 else 0
    print(f"\n  {mname}:")
    print(f"    NE/day (mean)  : {NE_day:.3f}")
    print(f"    NT/day (mean)  : {NT_day:.2f}")
    print(f"    S-bar (NT/NE)  : {sbar:.2f} cells")
    print(f"    Total obj-days : {NE_total:,}")

# ── 4. NE and NT per month per size class ────────────────────────────────────
print("\n" + "=" * 55)
print("4. NE PER MONTH BY SIZE CLASS")
print("=" * 55)

print(f"\n  {'Month':>6}  {'Sub-ref(5-15)':>15}  {'Medium(16-90)':>15}  {'Large(>=91)':>13}")
for mnum, mname in month_names.items():
    mdf      = df[df['month'] == mnum]
    n_days_m = times_pd[(pd.DatetimeIndex(times_pd).month == mnum)].shape[0]
    ne_sub   = ((mdf['size'] >= 5)  & (mdf['size'] < 16)).sum() / n_days_m
    ne_med   = ((mdf['size'] >= 16) & (mdf['size'] < 91)).sum() / n_days_m
    ne_lrg   = (mdf['size'] >= 91).sum() / n_days_m
    print(f"  {mname:>6}  {ne_sub:>15.3f}  {ne_med:>15.3f}  {ne_lrg:>13.3f}")

# ── 5. Yearly NT and NE (>=16 cells only, for trend analysis) ────────────────
print("\n" + "=" * 55)
print("5. YEARLY NT AND NE (>=16 cells, for S-bar trend)")
print("=" * 55)

df_trend = df[df['size'] >= 16]
print(f"\n  {'Year':>6}  {'NE':>8}  {'NT':>10}  {'S-bar':>8}")
for yr in range(1979, 2025):
    ydf  = df_trend[df_trend['year'] == yr]
    ne   = len(ydf)
    nt   = ydf['size'].sum()
    sbar = nt / ne if ne > 0 else 0
    print(f"  {yr:>6}  {ne:>8}  {nt:>10,}  {sbar:>8.2f}")

# ── 6. Quick sanity: mean NE and NT per JJAS day ─────────────────────────────
print("\n" + "=" * 55)
print("6. OVERALL DAILY MEANS")
print("=" * 55)
jjas_days = n_days
print(f"\n  Mean NE/day (>=5 cells)  : {len(df)/jjas_days:.3f}")
print(f"  Mean NT/day (>=5 cells)  : {df['size'].sum()/jjas_days:.2f}")
print(f"  Mean NE/day (>=16 cells) : {len(df[df['size']>=16])/jjas_days:.3f}")
print(f"  Mean NT/day (>=16 cells) : {df[df['size']>=16]['size'].sum()/jjas_days:.2f}")
print(f"  Briefing expected NE/day : 2.434")
print(f"  Briefing expected NT/day : 25.93")

Total ERE object-days (>= 5 cells): 6,579
Date range: 19790609 to 20240929

1. OVERALL SIZE DISTRIBUTION (cells per ERE object)

    Size range     Count        %   mean NT contrib
           5-7     1,807    27.5%               5.8
          8-10     1,125    17.1%               8.9
         11-15     1,123    17.1%              12.8
         16-25     1,104    16.8%              19.7
         26-50       903    13.7%              35.4
         51-90       374     5.7%              65.1
        91-150       118     1.8%             113.4
       151-250        25     0.4%             180.6
       251-500         0     0.0%               0.0
          >500         0     0.0%               0.0

  Total objects : 6,579
  Total NT cells: 130,929
  Mean size     : 19.90 cells
  Median size   : 12.0 cells
  Max size      : 243 cells
  75th pct size  : 23 cells
  90th pct size  : 45 cells
  95th pct size  : 62 cells
  99th pct size  : 120 cells

2. THREE SIZE CLASSES (briefing thresholds)

  